In [27]:
import torch
from torch import nn
import numpy as np
import requests
import torch_directml

In [28]:
SEQ_LENGTH = 100
BATCH_SIZE = 64
EMBEDDING_DIM = 128
HIDDEN_SIZE = 256
LEARNING_RATE = 1e-3
NUM_EPOCHS = 10
NUM_LAYERS = 2
DEVICE = "cpu"
CLIP = 5

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

In [29]:
response = requests.get(URL)
text = response.text
text = text[:100000]

chars = tuple(set(text))
int2char = dict(enumerate(chars))
char2int = {ch: i for i, ch in int2char.items()}

encoded_text = np.array([char2int[ch] for ch in text])

VOCAB_SIZE = len(chars)

In [30]:
def get_batches(arr, batch_size, seq_length):
    batch_size_total = batch_size * seq_length
    n_batches = len(arr) // batch_size_total
    arr = arr[:n_batches * batch_size_total]

    arr = arr.reshape((batch_size, -1))

    for n in range(0, arr.shape[1], seq_length):
        x = arr[:, n:n+seq_length]
        y = np.zeros_like(x)

        try:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, n+seq_length]
        except IndexError:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, 0]

        yield x, y


batches = get_batches(encoded_text, BATCH_SIZE, SEQ_LENGTH)
x_sample, y_sample = next(batches)

In [31]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        lstm_out, hidden = self.lstm(embedded, hidden)
        lstm_out = lstm_out.reshape(-1, self.hidden_size)
        output = self.fc(lstm_out)

        return output, hidden

    def init_hidden(self, batch_size, device):
        return (torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device),
                torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device))

model = CharLSTM(vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS).to(DEVICE)
print(model)

CharLSTM(
  (embedding): Embedding(61, 128)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True)
  (fc): Linear(in_features=256, out_features=61, bias=True)
)


In [32]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

model.train()
for epoch in range(NUM_EPOCHS):
    h = model.init_hidden(BATCH_SIZE, DEVICE)

    for x, y in get_batches(encoded_text, BATCH_SIZE, SEQ_LENGTH):
        inputs = torch.from_numpy(x).long().to(DEVICE)
        targets = torch.from_numpy(y).long().to(DEVICE)

        h = tuple([each.data for each in h])

        optimizer.zero_grad()
        output, h = model(inputs, h)
        loss = criterion(output, targets.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        optimizer.step()

    print(f"Epoch: {epoch+1}/{NUM_EPOCHS}, Loss: {loss.item():.4f}")

Epoch: 1/10, Loss: 3.2803
Epoch: 2/10, Loss: 3.0647
Epoch: 3/10, Loss: 2.7634
Epoch: 4/10, Loss: 2.5086
Epoch: 5/10, Loss: 2.3515
Epoch: 6/10, Loss: 2.2489
Epoch: 7/10, Loss: 2.1668
Epoch: 8/10, Loss: 2.0961
Epoch: 9/10, Loss: 2.0290
Epoch: 10/10, Loss: 1.9696
